In [89]:
using Pkg
Pkg.activate("..")
using Revise

  Activating project at `~/bslLD`


In [90]:
using bslLD, Plots, Statistics
bslLD.greet()

bslLD.use_cuda!()

Hello World!

In [117]:
grid =  bslLD.Grid([-5.0,-4.0],[5.0,4.0],[64,65],0.005,5000,1)

initFuncv(v)= exp(-(v+2)^2 / 2) / sqrt(2*pi)+ exp(-(v-2)^2 / 2) / sqrt(2*pi)
f = bslLD.Distribution(grid, 0.001,initFuncv=initFuncv);
initFuncv(v) = exp(-v^2 / 2) / sqrt(2*pi)

e = bslLD.empty_vectorfield(grid);


In [118]:
function step(f,grid)
    bslLD.advectX!(f,grid)
    ex = @. 0.05 * sin(2pi * grid.xaxes[1] / grid.max[1])
    e = bslLD.VectorField([collect(ex)])
    bslLD.advectV!(f,grid,e)
end

function stepSelfConsitent(f,grid)
    bslLD.advectX!(f,grid)
    rho = bslLD.compute_density(f,grid)
    phi = -1*bslLD.adiabatic(rho,grid)
    e = bslLD.compute_e(phi,grid)
    bslLD.advectV!(f,grid,e)
end

stepSelfConsitent (generic function with 1 method)

In [119]:
step(f,grid)

In [120]:
rhodiag = []
fdiag = []


for i in grid.itime
    stepSelfConsitent(f,grid)
    if i%10==0
        push!(fdiag,f.data[:,:])
        push!(rhodiag, bslLD.compute_density(f,grid).data[:])
    end
end


In [ ]:
num_frames = size(fdiag, 1)
frames_to_plot = 1:num_frames

animation = @animate for i in frames_to_plot
    heatmap(transpose(Array(fdiag[i])),
        title = "Frame $i",
        xlabel = "x",
        ylabel = "v",
    )

end
gif(animation, "fdiag_heatmap_animation.gif", fps = 10)

In [ ]:
plot(Array(rhodiag[10]))


In [ ]:
plot(map(x->(mean((x.-mean(x)).^2)), Array.(rhodiag)), yscale=:log10)